In [ ]:
from google.colab import drive
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/Colab Notebooks/AI/'

# 1. 토큰화 (Tokenization)

- 텍스트를 숫자로 변환하는 과정
- 딥러닝 모델은 `안녕하세요`와 같은 문자를 직접 이해할 수 없음.
- 오직 숫자만을 입력으로 받을 수 있으므로, 텍스트를 모델이 이해할 수 있는 숫자 형태로 변환하여야 함.
- **토큰 ID 시퀀스**로 데이터를 변환하는 역할을 **토크나이저**가 수행

## 1-1. 데이터 준비: 말뭉치(Corpus)구축

- 토크나이저를 생성하기 위해서 가장 먼저 고려해야 할 것은 `어떤 단어를 어떻게 나눌 것인가?`에 대한 기준을 학습시켜야 함.
- 학습이 필요하다 → 학습할 데이터가 필요하다.

### 1-1-1. 데이터 다운로드 및 확인

- `!wget [URL]`
    - URL이 가리키는 파일을 현재 작업 디렉토리에 다운로드 받음.
    - 파일 이름은 일반적으로 URL에 지정된 파일 이름을 따름.

In [ ]:
!wget https://github.com/e9t/nsmc/raw/master/ratings.txt

### 1-1-2. 데이터 정제 및 학습용 파일 생성

1. 데이터 정제
    - `ratings.txt` 원본 파일에는 **리뷰 텍스트** 외에 ID, 레이블 등 불필요한 정보나 비어있는 행 (결측치)도 포함되어 있음.
    - 이러한 데이터들의 경우 모델 학습에 방해되므로 전처리 과정이 필요
    1. pandas를 활용하여 결측치 제거
    2. 리뷰 내용이 담긴 열만 추출
    3. 추출한 텍스트를 `줄바꿈`으로 연결하여 새로운 텍스트 파일로 저장 (원본 보장)

In [ ]:
import pandas as pd

# 데이터 불러오기
# data = pd.read_csv('ratings.txt', sep='\t')
data = pd.read_csv(base_path + 'ratings.txt', sep='\t')
# 데이터 정보 확인
print(data.info())  # ID, document, label
print(data.head())  # 상위 5개 데이터 확인

# 결측치 제거
data = data.dropna()
# 리뷰 내용이 담긴 열만 추출: 'document'
data = data['document']

# 추출한 텍스트를 줄바꿈으로 연결하여 새로운 텍스트 파일로 저장
data.to_csv('nsmc.txt', index=False, header=False, sep='\n')

2. 추출한 데이터 정보 확인

In [ ]:
# 추출한 데이터 정보 확인
# with open('nsmc.txt', 'r') as file:
with open(base_path + 'nsmc.txt', 'r') as file:
    lines = file.readlines()
    print(f"총 리뷰 개수: {len(lines)}")
    print("샘플 리뷰:")
    for line in lines[:5]:  # 상위 5개 리뷰 출력
        print(line.strip()) # 앞뒤 공백 제거


## 1-2. 토크나이저 학습 및 활용

- 준비된 말뭉치(`nsmc.txt`)를 바탕으로, 문장을 어떤 규칙에 따라 토큰으로 분해 할지 정의
- **단어 사전(Vocabulary)를 구축**

### 1-2-1. 토크나이저(Tokenizer) 종류

1. 단어/규칙 기반
    - 가장 직관적이고 간단한 방식. 딥러닝 모델 등장 전부터 사용
    
    | 종류 | 설명 | 특징 |
    | --- | --- | --- |
    | 공백 기반 토크나이저 | 단순히 `공백` 기준으로 텍스트를 단어 단위로 나눔 예: `I love apples` → [’I’, ‘love’, ‘apples’] | 가장 단순하고 빠름. 단점: 한국어와 같이 단어 경계가 명확하지 않은 언어는 적용하기 어려움. `저는 개발자가 되고 싶어요.’ → 저는? 개발자? 개발자가? 되다. 싶어요. 등… 경계가 모호 |
    | 형태소 분석기 | 단어를 의미의 최소 단위인 형태소로 분리 (주로 한국어에 사용) 예: `공부하고` → [’공부’, ‘+하’, ‘+고’] | 한국어 처리에 특히 강함. 단점: 특정 언어에 의존적. 형태소 분석기 성능에 따라 품질이 달라짐. |
2. 서브워드 기반
    - 현대 언어 모델(LLM)에서 가장 널리 사용되는 토큰화 방식
    - 단어와 문자 수준의 장점을 결합하여 **OOV 문제를 최소화**하고 어휘 크기를 효율적으로 관리
    - **OOV(Out-Of-Vocabulary)**
        - 전처리 된 단어 목록에 존재하지 않는 새로운 단어가 등장하는 상황
        - 신조어, 오타, 희귀 단어, 형태 변화 등에 의해 발생 할 수 있음.
        - ex) 공백 기반 토크나이저로 처리한 데이터에 `나` 에 대한 문자가 다음과 같이 등장 할 수 있음.
            - 나는, 나를, 나에게, 나의, 난 등.
        - ex) 하지만, 실제 문장에서는 `나`에 대한 문자 중, 다음과 같은 상황도 있을 수 있음.
            - 나만, 나, 나에 등
        - 이러한 경우, 사전에 없는 단어가 등장하여 학습하지 않은 단어처럼 처리될 수 있음.
    
    | 종류 | 설명 | 특징 |
    | --- | --- | --- |
    | BPE (Byte Pair Encoding) | 가장 빈번하게 등장하는 문자 쌍을 찾아 새로운 토큰으로 병합 | 간단하고 효율적. GPT-3/4등 OpenAI 계열 모델에서 주로 사용 |
    | **WordPiece** | 통계적 확률을 기반으로 가장 가능성이 높은 하위 문자열 쌍을 병함 | 확률 기반으로, BPE보다 언어 가능성을 최대화. BERT등 구글 및 관련 계열의 모델에서 주로 사용 |

### 1-2-2. WordPiece 알고리즘과 토크나이저 초기화

- `BertWordPieceTokenizer`: BERT에서 사용된 WordPiece 방식의 토크나이저
    - BERT: 구글이 발표한 사전 훈련 언어 모델
    - 사전에 없는 새로운 단어**(Out-of-Vocabulary, OOV)**가 등장했을 때, 가능하다면, 아는 단어들의 조합으로 분해함.
        - ex) `자연어처리` 단어가 사전에 없지만, ‘자연어’와 ‘처리’ 는 사전에 있다면, **[‘자연어’, ‘##처리’]** 형태로 분해하여 처리함.
        - ex) 만약, 이렇게 분해조차 할 수 없다면 `[UNK]`(알 수 없음) 토큰으로 처리.
    - `##`: 앞 토큰에 이어서 붙인다는 의미의 접두사.

In [ ]:
from tokenizers import BertWordPieceTokenizer

# 아직 학습되지 않은, 비어있는 토크나이저 객체를 생성
tokenizer = BertWordPieceTokenizer(lowercase=False, strip_accents=False)

### 1-2-3. 토크나이저 학습

- `.train()`: 말뭉치 파일을 분석해서, 설정된 `vocab_size`에 맞춰 가장 효율적인 단어 사전을 생성
- `special_tokens`: 모델이 문장의 구조를 이해하거나 특정 과제를 수행하기 위해 사용하는 **특수 목적의 토큰**
    1. `[CLS]`(Classification): 문장의 시작을 의미. 문장 전체 정보를 요약하는 역할
    2. `[SEP]` (Separator): 두 문장을 구분
    3. `[PAD]` (Padding): 여러 문장을 한번에 처리(배치 처리)할 때, 길이를 맞춰주기 위해 짧은 문장 뒤에 채워 넣음.
        - ex) 배치 사이즈가 2일 때, 2개의 데이터. 
        [’안녕’, ‘하세요.’] 문장과 [’저는’, ‘개발자’, ‘입니다.’] 문장을 한 번에 처리한다면,
        [’안녕’, ‘하세요.’, ‘[PAD]’]  와 같이 처리
    4. `[UNK]` (Unknown): 사전에 없으면서 WordPiece로도 분해할 수 없는 단어를 위해 사용.
    5. `[MASK]` (Masking): BERT의 사전 학습(Pre-training) 시, 단어를 가리는 용도로 사용.

In [ ]:
# 토크나이저 학습
# vocab_size: 토큰 사전의 크기
    # 이 크기는 성능과 효율성 사이의 균형을 맞추는 데 중요
# min_frequency: 단어가 토큰 사전에 포함되기 위한 최소 빈도 수
# special_tokens: 모델에서 특별한 의미를 가지는 토큰들
tokenizer.train(
    # files='nsmc.txt',
    files=base_path + 'nsmc.txt',
    vocab_size=30000,
    min_frequency=2,
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"],
)

In [ ]:
# 토크나이저의 어휘 사전 크기 확인
print(f"어휘 사전 크기: {tokenizer.get_vocab_size()}")

# 토크나이저의 어휘 사전 무작위 10개 단어 확인
# 재현성 위해 시드 고정
import random
random.seed(42)

vocab = tokenizer.get_vocab()
sorted_vocab = sorted(vocab.items(), key=lambda x: x[1])  # 인덱스 기준으로 정렬
print("어휘 사전 무작위 10개 단어:")
for token, index in random.sample(sorted_vocab, 10):
    print(f"{index}: {token}")

### 1-2-4. 토큰화 실행 및 결과 확인

- 학습이 완료된 토크나이저는 어떤 문장이든 토큰 시퀀스와 ID 시퀀스로 변환 가능
- `.encode`: 사용자가 입력한 텍스트를 BERT 모델이 이해하고 처리할 수 있는 숫자 형태로 변환
    1. **토큰화:** 입력 문장을 토큰 시퀀스로 분할 (WordPiece 규칙 적용)
    2. **정수 인코딩:** 각 토큰을 미리 학습된 사전에 있는 고유한 정수 ID로 변환

In [ ]:
text = "나는 개발자 입니다!"
encoded = tokenizer.encode(text)

# .tokens: 분리된 토큰들의 리스트
print('토큰화 결과 :', encoded.tokens)
# 출력: ['나는', '개발', '##자', '입니다', '!']

# .ids: 각 토큰에 매핑된 고유 정수 ID의 리스트
print('정수 인코딩 :', encoded.ids)
# 출력: [2371, 9227, 1061, 3384, 5]